In [3]:
import json
import os
try:
    from backend.utils import load_json
except ModuleNotFoundError:
    print("Updating working directory....")
    os.chdir("../")
    print(os.getcwd())
    from backend.utils import load_json
    
from tqdm import tqdm 
from backend.utils import load_response_dict
from backend.dataset_utils import organize_dataset
from backend.nli_labels import ExNLILabel
import math

import matplotlib.pyplot as plt
import numpy as np


In [ ]:
from uncertainty import remove_invalid, sort_correct_wrong, construct_key_phrase

SyntaxError: trailing comma not allowed without surrounding parentheses (15957959.py, line 1)

In [ ]:
def find_relevant_logprobs(response, return_first=True, structure=None):
    # set up stuff
    
    prediction = response["prediction"]
    key_phrase = construct_key_phrase(prediction, structured_field=structure)
    # print("=== STAGE 1: SETUP")
    # print(f"Key Phrase: {key_phrase}")

    # find key phrase
    logprobs = response["logprobs"]
    token_pos = find_relevant_tokens(logprobs, key_phrase)
    # print(f"Positions: {token_pos}")
    # print(f"Phrase: {phrase_from_tokens(logprobs, token_pos)}")

    # find prediction phrase
    # print("=== STAGE 3: ISOLATE PREDICTION")
    key_phrase_tokens = logprobs[token_pos[0]:token_pos[-1]+1]
    # print([kpt["token"] for kpt in key_phrase_tokens])
    prediction_pos = find_relevant_tokens(key_phrase_tokens, prediction)
    # print(f"""Positions before correct: {prediction_pos}""")
    prediction_pos = [pp + token_pos[0] for pp in prediction_pos]
    # print(f"""Positions: {prediction_pos}
    if return_first:
        relevant_logprobs = [logprobs[pp] for pp in prediction_pos]
        return relevant_logprobs[0]
    else:
        return logprobs[prediction_pos[0]]
def logprob_to_prob(logprob, temp=1.):
    return math.exp(logprob / temp)

def get_probs(logprobs, temp=1.):
    # finds probabilities for a single token and its top logprobs
    for v in logprobs:
        # print(v)
        v["prob"] = logprob_to_prob(v["logprob"], temp=temp)
        top_logprobs = v["top_logprobs"]
        for token in top_logprobs:
            token["prob"] = logprob_to_prob(token["logprob"], temp=temp)
        # print(v["prob"])


def calculate_probs(responses, structure=None):
    for doc_id, doc_responses in tqdm(responses.items(), desc="Probabilities"):
        for hypo_id, hypo_response in doc_responses["annotation_sets"][0]["annotations"].items():
            logprobs = hypo_response["logprobs"]
            get_probs(logprobs, temp=1.)
            pred_tokens = find_relevant_logprobs(hypo_response, structure=structure)
            hypo_response["pred_tokens"] = pred_tokens

In [ ]:
dataset = "data/test.json"

# load dataset
dataset = load_json(dataset)
dataset = organize_dataset(dataset)



In [ ]:
cleaned_up = []
response_dir = [
    "out/responses/config_1/llama3.1_8b/narendra_nli_nonbinary/seed0",
    "out/responses/config_1/llama3.1_8b/narendra_nli_nonbinary/seed1",
    # "out/responses/config_1/gpt-oss_20b/narendra_nli_nonbinary/seed42"
]

for fp in response_dir:
        responses = load_response_dict(fp)
        remove_invalid(responses)
        cleaned_up.append(responses)

Loading responses: 100%|██████████| 123/123 [00:03<00:00, 39.66it/s]


Removed 0 invalid answers!


Loading responses: 100%|██████████| 123/123 [00:10<00:00, 11.79it/s]

Removed 0 invalid answers!


In [ ]:
def find_relevant_tokens(logprobs, key_phrase):
    # print("Assuming direct control")
    clean_keyphrase = "".join(key_phrase.split()) #remove whitespace to avoid issues
    token_buffer = []
    # start checking from the end of the list to reduce searching time
    for i, logprob in list(reversed(list(enumerate(logprobs)))):
        buffer_log = ""
        token = logprob["token"]
        clean_token = "".join(token.split())
        
        # check if the new subphrase can be added to buffer
        if len(token_buffer) > 0:
            # print("good", len(token_buffer))
            for j, tb in enumerate(token_buffer):
                pos, string = tb # list of token indices and the string they form
                subphrase = "".join((clean_token + string).split()) # phrase to be evaluated
                # print(subphrase, "//", clean_keyphrase)
                # print(token_buffer)

                new_pos = pos.copy()
                new_pos.insert(0, i)
                
                if clean_keyphrase in subphrase: # clean keyphrase is in the subphrase, a match is found (do "in" instead of "eq" because tokens can mass "eq" up)
                    return new_pos
                elif subphrase in clean_keyphrase: # break after this, because it alters the token buffer shape
                    
                    
                    # print(token_buffer, j, "->", end="")
                    token_buffer = token_buffer[j:] # clip the buffer to remove definite not-matches
                    
                    # update all in the rest of the buffer
                    for k, _ in enumerate(token_buffer):
                        k_pos, k_str = token_buffer[k]
                        k_pos.insert(0, i)
                        k_str = clean_token + k_str
                        token_buffer[k] = (k_pos, k_str)

                    # append newest token to buffer
                    token_buffer.append(([i], clean_token))
                    
                    # print(token_buffer)

                    break
                elif j+1 == len(token_buffer): # entire token buffer indexed without finding a match, clean token buffer with only the current token
                    # print("Next token")
                    buffer_line = f"{i} (DUMP): {clean_token} | {token_buffer[0][1]} \n"
                    buffer_log += buffer_line
                    token_buffer = [([i], clean_token)]
                    break # unnecessary but just to make it clear


        
        else:
            token_buffer = [([i], clean_token)]
    
    out_str = ""
    for lp in logprobs:
        out_str += lp["token"]
    print("==== CLEAN STUFF   ====")
    print(clean_keyphrase)
    print("==== OUTPUT STRING ====")
    print(out_str)
    print("====   BUFFER LOG  ====")
    print( buffer_log)
    error_msg = f"Key phrase '{key_phrase}' not found in logprobs tokens!"
    raise KeyError(error_msg)

# load and process responses
def big_fn(cleaned, structure):
    correct, wrong = {}, {}
    for responses in cleaned:

        # calculate probs
        calculate_probs(responses, structure=structure)

        # find correct/wrong
        c, w = sort_correct_wrong(dataset, responses)

        # complicated indexing to save on boilerplating 
        for big_dict, sub_dict in [(correct, c), (wrong, w)]:
            print(big_dict.keys(), sub_dict.keys())
            if len(big_dict) == 0:
                for k,v in sub_dict.items():
                    big_dict[k] = v
                print(big_dict.keys())
            else:
                for k,v in big_dict.items():
                    big_dict[k] = v + sub_dict[k]


big_fn(cleaned_up, structure=None)

Sorting: 100%|██████████| 123/123 [00:00<00:00, 24191.10it/s]


Num correct: 1073
Num wrong:   1018
dict_keys([]) dict_keys(['Not', 'Ent', 'Contr', 'N', '"No'])
dict_keys(['Not', 'Ent', 'Contr', 'N', '"No'])
dict_keys([]) dict_keys(['Ent', 'Contr', 'Not', '"No', 'Con', 'N'])
dict_keys(['Ent', 'Contr', 'Not', '"No', 'Con', 'N'])


Sorting: 100%|██████████| 123/123 [00:00<00:00, 27033.08it/s]

Num correct: 1021
Num wrong:   1070
dict_keys(['Not', 'Ent', 'Contr', 'N', '"No']) dict_keys(['Not', 'Ent', 'Contr', '"Not', 'N', '"No', '"N', 'E', 'C', '"E'])
dict_keys(['Ent', 'Contr', 'Not', '"No', 'Con', 'N']) dict_keys(['Ent', 'Not', '"E', 'Contr', 'C', 'N', '"No', '"Not', '"N'])


KeyError: 'Con'